In [1]:
!pip install -q transformers datasets accelerate evaluate

In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# Load IMDb dataset
raw_data = load_dataset("stanfordnlp/imdb")

# Smaller subset so it trains smoothly on CPU
train_data = raw_data["train"].shuffle(seed=42).select(range(400))
test_data = raw_data["test"].shuffle(seed=42).select(range(100))

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_text(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_data.map(tokenize_text, batched=True)
tokenized_test = test_data.map(tokenize_text, batched=True)


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

... (6 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [4]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def get_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=get_metrics,
)

trainer.train()


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.664592,0.660000,0.673077
2,No log,0.624449,0.720000,0.714286


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=50, training_loss=0.6544673156738281, metrics={'train_runtime': 1274.2934, 'train_samples_per_second': 0.628, 'train_steps_per_second': 0.039, 'total_flos': 26493479731200.0, 'train_loss': 0.6544673156738281, 'epoch': 2.0})

In [5]:
# Function to predict sentiment on new sentences
def predict_sentiment(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    pred_class = torch.argmax(outputs.logits, dim=1).item()
    label = "Positive" if pred_class == 1 else "Negative"
    print(f"Review: '{text}'\nPrediction: {label}\n")

# Testing with custom reviews
predict_sentiment("Honestly, the storyline was slow at first, but the ending totally saved it.")
predict_sentiment("I wasted two hours of my life watching this. Terrible acting.")



Review: 'Honestly, the storyline was slow at first, but the ending totally saved it.'
Prediction: Negative

Review: 'I wasted two hours of my life watching this. Terrible acting.'
Prediction: Negative



### Project Summary & Notes

**How Transfer Learning Was Applied:**
Instead of training a neural network from scratch—which would require millions of text samples and days of computing time—I used transfer learning with `distilbert-base-uncased`. Because DistilBERT was already pre-trained on massive amounts of English text (Wikipedia and BookCorpus), it already understands grammar, context, and word relationships. I added a binary classification head on top and fine-tuned those existing weights on a 400-sample subset of the IMDb dataset (chosen to train efficiently on CPU) for 2 epochs, improving validation accuracy from 66% to 72% (F1-score: 0.714).

**Power of Pre-Trained LLMs:**
* **High accuracy with small data:** Even with only 400 training reviews, the model reached 72% accuracy in just 2 epochs because of its pre-existing language understanding.
* **Practical accessibility:** Using a distilled model allowed me to fine-tune a real transformer architecture even without a dedicated GPU.

**Limitations:**
* **Compute time on CPU:** Training even a small subset on CPU still took around 20 minutes, showing how resource-heavy transformers are compared to classical models like Logistic Regression.
* **Input length limits:** Transformer models have strict token limits (truncated to 128 tokens here), meaning longer movie reviews lose context from the end of the text.
* **Subtlety and sarcasm:** While it catches clear positive and negative sentiment well, heavily sarcastic or nuanced reviews can still be misclassified.
